# Processamento Distribuído e Agregações Analíticas - Camada Gold (PySpark)
Consolidação analítica de KPIs de performance, volume acumulado e volatilidade média com Spark.


In [ ]:
import os
import pandas as pd

DATA_DIR = '../data/stocks'
silver_path = os.path.join(DATA_DIR, 'stocks_silver_refined.csv')
df_silver = pd.read_csv(silver_path)

try:
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import col, avg, max, min, sum, round as spark_round
    
    spark = SparkSession.builder.appName("StockAnalysisGold").master("local[*]").getOrCreate()
    df_spark = spark.createDataFrame(df_silver)
    
    # Agregações executivas para consumo de BI e Agentes de IA
    df_gold = df_spark.groupBy("ticker").agg(
        spark_round(avg("close"), 2).alias("avg_close_price"),
        spark_round(max("high"), 2).alias("max_52w_approx"),
        spark_round(min("low"), 2).alias("min_52w_approx"),
        spark_round(avg("volatility_7d"), 4).alias("avg_volatility_pct"),
        sum("volume").alias("total_volume_traded")
    ).orderBy("ticker")
    
    df_gold.show()
    gold_df_pandas = df_gold.toPandas()
except Exception as e:
    print(f"Fallback analítico sem Spark Cluster ativo: {e}")
    gold_df_pandas = df_silver.groupby('ticker').agg(
        avg_close_price=('close', 'mean'),
        max_price=('high', 'max'),
        min_price=('low', 'min'),
        avg_volatility_pct=('volatility_7d', 'mean'),
        total_volume=('volume', 'sum')
    ).reset_index()

gold_path = os.path.join(DATA_DIR, 'stocks_gold_analytics.csv')
gold_df_pandas.to_csv(gold_path, index=False)
print(f"Camada Gold Analytics finalizada e persistida em: {gold_path}")
gold_df_pandas.head()
